# 音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜

### 第4章：予測子の扱い：コントラストコーディングと中心化


ここに掲載されているコードは、
『音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜』の付録である。
本書の巻末に載っているGitHubリポジトリのリンクから、「Open in Colab」のバッジで開いている。

「このノートブックはGoogleが作成したものではありません」という警告は、外部（GitHub）から
読み込むノートブック全般に出る定型の警告である。内容を確認のうえ実行してよい。
書き込んだ内容を残したいときは、「ファイル」→「ドライブにコピーを保存」で自分のGoogle Driveに
保存する。保存したコピーは自由に変更・実行・保存してよい。

**説明はすべてコード中の#コメントとセル内のmarkdownに書いてある。**
コピペ元を別に参照する必要はなく、このノートブックを上から実行しながら読めば完結する。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で言語を **R** に切り替えること。
既定のPythonランタイムのままだと以降のセルがすべてエラーになる。


参照論文: Schad, Vasishth, Hohenstein & Kliegl (2020), *Journal of Memory and Language*。

In [ ]:
# ---- 共通データセットの読み込み ----
# 第1章と全く同じ手順・同じ乱数seedでVOTデータを作り直す。
# 「本ごとに違うデータ」を避けるためだが、外部URLから読み込むのではなく
# その場で再生成する設計にしてある——理由は、配布用ノートブックが
# 非公開のGitHubリポジトリのURLに依存すると、読者の環境からは
# アクセスできず404になってしまうため（実際にこの問題が起きて修正した）。
# コードを自分の環境で動かす分には、この方式のほうが「ネットワーク不要で
# 必ず同じ結果になる」という利点もある。
if (!requireNamespace("tidyverse", quietly = TRUE)) install.packages("tidyverse", repos = "https://cloud.r-project.org")
library(tidyverse)
# ---- 図の日本語文字化け対策・文字サイズの既定値 ----
# Colabの既定フォントには日本語グリフが含まれないため、図のタイトルや軸ラベルが
# 文字化け（豆腐記号）して表示される。Google Fontsの日本語フォントを取得し、
# "notosans"という名前で登録して使う（要ネット接続。Colabでは常に使える）。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")  # "sans"などの予約名は上書きできないため専用名で登録
showtext_auto()
showtext_opts(dpi = 100)                     # Colabのインライン画像解像度(100dpi)に合わせる
par(family = "notosans")                     # base Rの作図（mclust・itsadugなど）にも適用する
theme_set(theme_minimal(base_size = 14, base_family = "notosans"))  # ggplotの既定テーマ


set.seed(42)  # 乱数のseedを固定する。同じ数字なら誰が実行しても同じ結果になる
n_speaker <- 10                    # 話者数
n_word    <- 20                    # 単語数（各話者が全単語を1回ずつ発話する）
n_obs     <- n_speaker * n_word    # = 200行

# 何行目がどの話者・どの単語かを表す添字。あとで話者ごと・単語ごとの
# 値を引くのに使う（speaker_intercept[speaker_idx] のように書ける）。
speaker_idx <- rep(1:n_speaker, each  = n_word)
word_idx    <- rep(1:n_word,    times = n_speaker)

# 1行ごとにvoiced/voicelessを交互に並べる。話者1人あたり20行なので、
# どの話者を見ても必ずvoiced10回・voiceless10回が含まれる
# ——これは第5章で「話者ごとにconditionの効果を推定する」ために必須の設計。
condition_vec <- rep(c("voiced", "voiceless"), times = n_obs / 2)

# ---- 話者差・単語差を「先に」作っておく ----
# 実際の音声データでは、同じ条件でも話者によってVOTの平均が違い（声道長・
# 発話速度・方言）、単語によっても違う（後続母音・語長・頻度）。
# 擬似データでもこの構造を明示的に作っておかないと、第5章以降で使う
# 混合効果モデルが「推定すべきばらつきが最初から存在しない」データを
# 相手にすることになり、分散推定値がすべて0（singular fit）になってしまう。
speaker_intercept <- rnorm(n_speaker, mean = 0, sd = 6)
# 話者ごとのVOTの底上げ量。どの条件にも一律にかかる（＝ランダム切片）。
speaker_slope <- rnorm(n_speaker, mean = 0, sd = 10)
# 話者ごとの「有声と無声をどれだけ大きく作り分けるか」の差（＝ランダム傾き）。
# 話者間のばらつきは無声（VOTが長い側）で特に大きいことが知られているので、
# voiceless条件にだけ加える。
word_intercept <- rnorm(n_word, mean = 0, sd = 3)
# 単語ごとの癖（＝項目のランダム切片）。

vot_data <- tibble(
  speaker   = paste0("S", sprintf("%02d", speaker_idx)),
  word      = paste0("word_", word_idx),
  gender    = rep(rep(c("M", "F"), each = n_speaker / 2), each = n_word),
  condition = condition_vec,
  # VOTは「固定効果 + 話者差 + 単語差 + 残差」を足し合わせて作る。
  # これは第5章で当てはめる混合効果モデルの式そのものであり、
  # モデルが推定した分散をこの生成時の値と見比べられるようにしてある。
  VOT = 20 +                                              # 有声条件の基準値
    if_else(condition == "voiceless", 55, 0) +            # 無声条件の上乗せ（→約75ms）
    speaker_intercept[speaker_idx] +                      # 話者のランダム切片
    if_else(condition == "voiceless",                     # 話者のランダム傾き
            speaker_slope[speaker_idx], 0) +
    word_intercept[word_idx] +                            # 単語のランダム切片
    # 残差。無声条件のほうがばらつきが大きい（実際のVOTデータもそうなっている）。
    # 第3章のモデル診断で「等分散性が崩れている」例として使う。
    rnorm(n_obs, mean = 0, sd = if_else(condition == "voiced", 4, 8)),
  F0 = rnorm(n_obs, mean = 130, sd = 15)  # F0はここでは条件と無関係に生成
)

vot_data <- vot_data %>%
  mutate(
    # 文字列のままだとRが「順序のない値」として扱うため、
    # 話者・単語・性別・条件をすべてfactor（因子）型に変換しておく。
    # 特にconditionはlevels=でvoiced/voicelessの順序を明示し、
    # 以降のモデルの切片が「voiced群の平均」になるよう固定する。
    speaker   = factor(speaker),
    word      = factor(word),
    gender    = factor(gender),
    condition = factor(condition, levels = c("voiced", "voiceless"))
  )

glimpse(vot_data)  # 型と最初の数行を確認。数値であるべき列が<chr>のままなら要注意

# --- 実行結果 ---
# Rows: 200
# Columns: 6
# $ speaker   <fct> S01, S01, S01, S01, S01, S01, S01, S01, S01, S01, S01, ...
# $ word      <fct> word_1, word_2, word_3, word_4, word_5, word_6, word_7, ...
# $ gender    <fct> M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, ...
# $ condition <fct> voiced, voiceless, voiced, voiceless, voiced, voiceless, ...
# $ VOT       <dbl> 28.129829, 88.042064, 30.742652, 94.104833, 28.438207, ...
# $ F0        <dbl> 119.0617, 144.9710, 148.8772, 148.7330, 109.2904, ...
#
# 【出力の読み方】
# ・Rows: 200 / Columns: 6 が行数と列数。10話者×20語=200行になっているかを
#   まず確認する。ここが合っていなければ、以降の分析はすべて無意味になる。
# ・各行の <fct> <dbl> が**型**である。<fct>はfactor（水準を持つカテゴリ）、
#   <dbl>は小数を含む数値。speaker・word・gender・conditionがすべて<fct>、
#   VOTとF0が<dbl>になっていれば正しい。
#   **VOTが<chr>（文字列）になっていたら、そこで止まって読み込みを見直す。**
#   文字列のままでは平均もモデルも計算できず、しかもエラーではなく
#   「変な結果」として静かに進んでしまうことがある。
# ・conditionの並びがvoiced, voiceless, voiced, voiceless...と1行おきに
#   交互になっている。これは意図した設計どおり（どの話者も両条件を10回ずつ持つ）。
# ・speakerが先頭でS01ばかり並ぶのは、話者ごとに20行がまとまっているため。
#   行の並び順自体は分析結果に影響しない（第7章のAR(1)を使うときだけ例外）。


■ treatment codingの確認（既定）：

**実際の研究例**：Schad, Vasishth, Hohenstein & Kliegl (2020, *Journal of Memory and Language*) は、心理言語学・音声学の実験研究の多くが、Rの既定であるtreatment codingを無自覚に使い、それが単純主効果しか推定していないことに気づかないまま交互作用を解釈していると指摘した。

Rの既定は treatment coding（基準水準との差）。切片は基準水準の平均になる。

In [ ]:
contrasts(vot_data$condition)  # 現在のコーディング方式を確認
lm(VOT ~ condition, data = vot_data) %>% summary()
# 切片(Intercept)はcondition="voiced"（基準水準）の平均VOT。
# conditionvoicelessの係数は「voiceless群がvoiced群よりどれだけ長いか」の差。

# --- 実行結果（抜粋）---
#           voiceless
# voiced            0
# voiceless         1
#
# Coefficients:
#                    Estimate Std. Error t value Pr(>|t|)
# (Intercept)          22.867      1.303   17.55   <2e-16 ***
# conditionvoiceless   52.500      1.842   28.50   <2e-16 ***
#
# 【出力の読み方】
# ・最初の小さな表が**コントラスト行列**である。行が水準、列が作られる
#   予測変数（ダミー変数）。voiced行が0、voiceless行が1なので、
#   「voicedのとき0、voicelessのとき1になる変数を1本作る」という意味になる。
#   列名が voiceless なので、係数名も conditionvoiceless になる。
# ・R既定のこの方式を**treatment coding（ダミーコーディング）**と呼ぶ。
#   水準がk個なら列はk-1本作られ、行が全部0になる水準（ここではvoiced）が
#   **基準水準**になる。基準はfactorの最初の水準で、第1章で
#   levels = c("voiced","voiceless") と明示したのはこれを固定するためである。
# ・したがって (Intercept) 22.867 は「ダミー変数が0のときの予測値」＝
#   **voiced群の平均**であり、全体の平均ではない。
# ・conditionvoiceless 52.500 は「ダミーが0から1になったときの増分」＝
#   voiceless群の平均 − voiced群の平均。実際 75.367 − 22.867 = 52.500。
# ・**この読み方は水準が2つのうちは直感的だが、交互作用が入ると崩れる。**
#   基準水準に限った話（単純主効果）になってしまうためで、その解決が
#   次のセルから始まるsum codingである。


■ sum codingへの変更：

Schad et al. (2020) が推奨するように、単純主効果ではなく主効果そのものを解釈したい場合は sum coding（±0.5や±1）に変える。

In [ ]:
vot_data_sum <- vot_data %>%
  mutate(condition_sum = C(condition, sum))  # C()でコントラストを明示的に上書きする

lm(VOT ~ condition_sum, data = vot_data_sum) %>% summary()
# sum codingでは切片が「voiced群とvoiceless群を単純平均した値」になる。
# 2水準だけなら係数の大きさはtreatment codingの半分になる（符号の意味も変わる）。

# --- 実行結果（抜粋）---
# Coefficients:
#                Estimate Std. Error t value Pr(>|t|)
# (Intercept)     49.1170     0.9212   53.32   <2e-16 ***
# condition_sum1 -26.2501     0.9212  -28.50   <2e-16 ***
#
# 【出力の読み方】
# ・sum coding（偏差コーディング）ではvoicedに+1、voicelessに-1を割り当てる。
#   0と1ではなく**+1と-1で対称に配置する**ことが要点である。
# ・(Intercept) 49.1170 は2群の平均をさらに平均した値
#   （22.867 + 75.367）/ 2 = 49.117。treatment codingの切片22.867が
#   「voiced群の平均」だったのに対し、こちらは**全体の水準**を表す。
#   この「切片が全体平均になる」ことが、交互作用を含むモデルで効いてくる。
# ・condition_sum1 −26.2501 は、treatment codingの52.500のちょうど**半分**に
#   符号を逆にした値である（52.500 / 2 = 26.25）。
#   係数が「基準からの差」ではなく「全体平均からのずれ」を表すため半分になる。
#   符号が負なのは、+1を割り当てられたvoicedが全体平均より26.25ms**短い**から。
#   ★係数名が condition_sum1 と抽象的になり、どちらの水準が+1か出力から
#   読み取れない点に注意する。contrasts(vot_data_sum$condition_sum) で
#   必ず確認する習慣をつける。
# ・**t値・p値・R-squared・F統計量はtreatment codingのときと完全に同じ**
#   （t = -28.50、R² = 0.804、F = 812）。コーディングを変えても
#   モデルの当てはまり自体は1ミリも変わらない。変わるのは
#   **係数をどう言葉にするか**だけである。ここを取り違えて
#   「sum codingにしたら有意になった」と考えてはいけない。


■ 交互作用のコントラストコーディング：

gender×conditionの交互作用をsum codingで組み、切片が
「全群の（不加重でない）平均」に対応することを確認する。

In [ ]:
vot_data_sum2 <- vot_data %>%
  mutate(
    condition_sum = C(condition, sum),
    gender_sum = C(gender, sum)
  )

lm(VOT ~ condition_sum * gender_sum, data = vot_data_sum2) %>% summary()
# treatment codingのまま交互作用モデルを読むと、主効果の係数は
# 「基準水準における単純主効果」であって全体の主効果ではない。
# sum codingならこの係数がそのまま主効果として解釈できる。

# --- 実行結果（抜粋）---
# Coefficients:
#                            Estimate Std. Error t value Pr(>|t|)
# (Intercept)                 49.1170     0.8863  55.416  < 2e-16 ***
# condition_sum1             -26.2501     0.8863 -29.616  < 2e-16 ***
# gender_sum1                 -2.2656     0.8863  -2.556 0.011341 *
# condition_sum1:gender_sum1   2.9874     0.8863   3.370 0.000904 ***
#
# 【出力の読み方】
# ・**本章の主張がこの1枚で確認できる。** 第3章でtreatment codingのまま
#   同じ交互作用モデルを当てはめたとき、genderM の p 値は 0.565 で、
#   ANOVA表のgender行の p = 0.011341 と食い違っていた。
#   sum codingにした今、gender_sum1 の p は **0.011341** となり、
#   ANOVA表の主効果と**完全に一致**する。
#   つまり「ANOVAと回帰で結果が違う」のではなく、
#   **treatment codingの係数が主効果ではなかった**だけである。
# ・条件も同様で、condition_sum1 の t = -29.616 は、両性別をまとめた
#   条件の主効果を表す。treatment codingの conditionvoiceless（t = 18.559）は
#   「F（基準水準）に限った条件の効果」でしかなかった。
# ・交互作用 condition_sum1:gender_sum1 の p = 0.000904 は
#   treatment codingのときと**変わらない**。交互作用項だけは
#   コーディングの取り方に依存しないためである。
#   **主効果はコーディングで意味が変わるが、交互作用は変わらない。**
# ・係数の大きさも整合する。treatment codingの交互作用 11.949 に対し、
#   sum codingでは 2.9874。ちょうど 11.949 / 4 = 2.987 である
#   （2要因それぞれで半分ずつになるため 2×2 = 4分の1）。
# ・実務上の指針: **交互作用を含むモデルで主効果を報告したいなら、
#   sum coding（またはその他の直交コントラスト）にしてから読む。**
#   treatment codingのまま「genderの主効果は有意でなかった」と書くと、
#   実際には「voiced条件におけるgenderの効果」を報告したことになる。


■ 概念をつかむ：係数は何と何の差か：

同じデータ・同じ式でも、コーディングを変えると切片と係数が指す平均の組み合わせが変わる。3水準の頻度帯freq(low/mid/high)で、treatment・sum・Helmertの3通りを当てはめ、lm()の係数と手計算の平均の差が一致することを確かめる。

In [ ]:
# 3水準の頻度帯 freq を作る（第4章の演習2と同じ手順）
word_number <- as.integer(sub("word_", "", as.character(vot_data$word)))
pair_id <- ceiling(word_number / 2)
band <- c("low", "mid", "high")[((pair_id - 1) %% 3) + 1]
vot_freq <- vot_data %>%
  mutate(
    freq = factor(band, levels = c("low", "mid", "high")),
    VOT = VOT + case_when(freq == "low" ~ 0, freq == "mid" ~ -6, freq == "high" ~ -12)
  )

# 各水準の平均を先に出す。以下の係数はすべてこの3つの組み合わせである。
m <- tapply(vot_freq$VOT, vot_freq$freq, mean)
m_low <- m[["low"]]; m_mid <- m[["mid"]]; m_high <- m[["high"]]
print(round(m, 3))
cat("3水準の平均の単純平均:", round(mean(m), 3),
    "  全200観測の平均:", round(mean(vot_freq$VOT), 3), "\n\n")

# 同じ式を3通りのコーディングで当てはめる
fit_tr <- lm(VOT ~ freq, data = vot_freq)                       # treatment（既定）
fit_su <- lm(VOT ~ C(freq, sum), data = vot_freq)               # sum
fit_he <- lm(VOT ~ C(freq, contr.helmert), data = vot_freq)     # Helmert

# lm の係数 と 手計算（平均の組み合わせ）を並べる
res <- rbind(
  data.frame(coding = "treatment", term = c("切片", "係数1", "係数2"),
             lm = unname(coef(fit_tr)),
             hand = c(m_low, m_mid - m_low, m_high - m_low),
             meaning = c("low", "mid - low", "high - low")),
  data.frame(coding = "sum", term = c("切片", "係数1", "係数2"),
             lm = unname(coef(fit_su)),
             hand = c(mean(m), m_low - mean(m), m_mid - mean(m)),
             meaning = c("3水準の平均", "low - 3水準の平均", "mid - 3水準の平均")),
  data.frame(coding = "Helmert", term = c("切片", "係数1", "係数2"),
             lm = unname(coef(fit_he)),
             hand = c(mean(m), (m_mid - m_low) / 2, (m_high - (m_low + m_mid) / 2) / 3),
             meaning = c("3水準の平均", "(mid - low)/2", "(high - (low+mid)/2)/3"))
)
res$lm <- round(res$lm, 3); res$hand <- round(res$hand, 3)
print(res, row.names = FALSE)

# 当てはまり自体はコーディングに依らない
cat("\nR2:", round(c(summary(fit_tr)$r.squared, summary(fit_su)$r.squared,
                    summary(fit_he)$r.squared), 4), "\n")

# --- 実行結果 ---
#    low    mid   high 
# 47.073 43.933 39.026 
# 3水準の平均の単純平均: 43.344   全200観測の平均: 43.717
#
#     coding  term     lm   hand                meaning
#  treatment  切片 47.073 47.073                    low
#  treatment 係数1 -3.139 -3.139              mid - low
#  treatment 係数2 -8.046 -8.046             high - low
#        sum  切片 43.344 43.344            3水準の平均
#        sum 係数1  3.729  3.729      low - 3水準の平均
#        sum 係数2  0.589  0.589      mid - 3水準の平均
#    Helmert  切片 43.344 43.344            3水準の平均
#    Helmert 係数1 -1.570 -1.570          (mid - low)/2
#    Helmert 係数2 -2.159 -2.159 (high - (low+mid)/2)/3
#
# R2: 0.0129 0.0129 0.0129
#
# 【出力の読み方】
# ・lm列と手計算列が3通りのコーディングすべてで一致している。係数は
#   モデルが決めた不思議な量ではなく、水準平均の組み合わせである。
# ・treatment codingの切片は基準水準lowの平均(47.073)、係数は基準との差である。
# ・sum codingとHelmertの切片は同じ43.344で、3水準の平均の単純平均になる。
#   各水準の語数が違う(low 80行、mid・high 60行)ため、全200観測の平均
#   43.717とは一致しない。切片は観測を数えた平均ではなく水準平均の平均である。
# ・Helmertの係数は差を水準数で割った値で、実際の差を報告するときは2倍・3倍する。
# ・R2は3通りとも同じ。コーディングを変えても当てはまりは変わらず、
#   変わるのは係数を言葉にする仕方だけである。


■ 連続予測子にも同じ問題がある：中心化とスケーリング：

切片は「すべての予測子がゼロのときの予測値」で決まる、という原則は連続量でも変わらない。条件とF0（連続量）の交互作用モデルで確かめる。

In [ ]:
vot_data_sum <- vot_data_sum %>% mutate(F0_c = F0 - mean(F0))

cat("=== 生のF0（未中心化）===\n")
m_raw <- lm(VOT ~ condition_sum * F0, data = vot_data_sum)
print(summary(m_raw)$coefficients)

cat("\n=== 中心化したF0 ===\n")
m_c <- lm(VOT ~ condition_sum * F0_c, data = vot_data_sum)
print(summary(m_c)$coefficients)

cat("\nraw R2:", summary(m_raw)$r.squared, " centered R2:", summary(m_c)$r.squared, "\n")

# --- 実行結果（抜粋）---
# === 生のF0（未中心化）===
#                       Estimate Std. Error  t value  Pr(>|t|)
# (Intercept)         53.777924   8.360502   6.4324  9.40e-10
# condition_sum1     -26.756031   8.360502  -3.2003  1.60e-03
# F0                  -0.035929   0.064039  -0.5610  5.75e-01
# condition_sum1:F0    0.004046   0.064039   0.0632  9.50e-01
#
# === 中心化したF0 ===
#                        Estimate Std. Error  t value   Pr(>|t|)
# (Intercept)          49.114809   0.925793  53.0516  3.20e-118
# condition_sum1      -26.230980   0.925793 -28.3335   3.13e-71
# F0_c                 -0.035929   0.064039  -0.5610   5.75e-01
# condition_sum1:F0_c   0.004046   0.064039   0.0632   9.50e-01
#
# raw R2: 0.8042661  centered R2: 0.8042661
#
# 【出力の読み方】
# ・生のF0（範囲89.5〜166.9Hz、平均129.8Hz）のままだと、切片53.78（SE 8.36）は
#   **F0=0Hzのとき**の予測値になる。声帯振動が無いことを意味するF0=0Hzは
#   実際のデータ範囲から遠く外れた外挿であり、標準誤差の大きさ（8.36）は
#   この外挿の不確実性をそのまま反映している。
# ・F0を平均で中心化すると、切片は49.11（SE 0.93）に変わる。これは
#   **平均的なF0における予測値**であり、標準誤差も9分の1近くまで小さくなる。
# ・**交互作用の係数（0.004046）・p値（0.9497）・R²（0.8043）は、
#   中心化してもまったく変わらない。** 中心化は線形変換なので当てはまり
#   そのものには影響せず、変わるのは切片と主効果の解釈だけである。


■ 演習：

1. treatment codingとsum codingで切片の意味がどう変わるか説明せよ。
2. Helmertコーディングを使い、3水準以上のカテゴリカル変数（例: word frequency band）を作って係数を解釈せよ。

■ 演習の解答：

1問目は前のセルの出力を言葉にする問題なので、答えはコメントに書いてある。2問目は3水準の因子を新しく作り、Helmertコントラストを当てて係数を読む。

In [ ]:
# 1. treatment codingとsum codingで切片の意味がどう変わるか
#
#    treatment coding : 切片 = 基準水準（voiced）の平均            → 22.867
#    sum coding       : 切片 = 各水準の平均をさらに平均した値      → 49.117
#
#    数字で確かめる。
mean_voiced    <- mean(vot_data$VOT[vot_data$condition == "voiced"])
mean_voiceless <- mean(vot_data$VOT[vot_data$condition == "voiceless"])

cat("voicedの平均               :", round(mean_voiced, 3), "\n")
cat("2群の平均の平均            :", round((mean_voiced + mean_voiceless) / 2, 3), "\n")
cat("treatment codingの切片     :", round(coef(lm(VOT ~ condition, data = vot_data))[1], 3), "\n")
cat("sum codingの切片           :", round(coef(lm(VOT ~ condition_sum, data = vot_data_sum))[1], 3), "\n")

# 2. 3水準のカテゴリカル変数を作り、Helmertコーディングで係数を解釈する。
#    word_1..word_20 を頻度帯（low / mid / high）に割り当てる。
#    ★割り当て方に一手間かける。第1章でconditionは語の奇数/偶数で決まるので、
#      単純に3語ずつ区切るとfreqとconditionが釣り合わなくなる（低頻度帯に
#      有声語が偏る等）。そこで「隣り合う2語＝有声1語+無声1語」を1組にして、
#      組ごとに頻度帯を割り当てる。こうすればどの帯も有声/無声が同数になる。
word_number <- as.integer(sub("word_", "", as.character(vot_data$word)))
pair_id <- ceiling(word_number / 2)                      # (1,2)→1, (3,4)→2, ...
band <- c("low", "mid", "high")[((pair_id - 1) %% 3) + 1]

# ここでは「頻度が高い語ほど発話が速く、VOTが短くなる」と仮定して擬似的な
# 効果を足す（実データを持っている人は自分の頻度情報に置き換える）。
vot_freq <- vot_data %>%
  mutate(
    freq = factor(band, levels = c("low", "mid", "high")),
    VOT = VOT + case_when(freq == "low" ~ 0, freq == "mid" ~ -6, freq == "high" ~ -12)
  )

table(vot_freq$condition, vot_freq$freq)  # 釣り合っているかを必ず確認する

# Helmertコントラストを割り当てる。
# 第1列: 水準2を水準1と比べる                  (mid  vs low)
# 第2列: 水準3を「水準1と2の平均」と比べる      (high vs low+midの平均)
contrasts(vot_freq$freq) <- contr.helmert(3)
contrasts(vot_freq$freq)

# conditionを入れずにfreqだけで回すと、条件による50msの差がまるごと
# 残差に化けてfreqの効果が埋もれる。統制すべき要因は必ずモデルに入れる。
summary(lm(VOT ~ condition + freq, data = vot_freq))

# 係数が本当に「水準平均の差」から作られているかを手計算で確かめる。
m_low  <- mean(vot_freq$VOT[vot_freq$freq == "low"])
m_mid  <- mean(vot_freq$VOT[vot_freq$freq == "mid"])
m_high <- mean(vot_freq$VOT[vot_freq$freq == "high"])
cat("freq1 に対応する値 (mid - low)/2            :", round((m_mid - m_low) / 2, 4), "\n")
cat("freq2 に対応する値 (high - (low+mid)/2)/3   :",
    round((m_high - (m_low + m_mid) / 2) / 3, 4), "\n")

# --- 実行結果（抜粋）---
# voicedの平均               : 22.867
# 2群の平均の平均            : 49.117
# treatment codingの切片     : 22.867
# sum codingの切片           : 49.117
#
#             low mid high
#   voiced     40  30   30
#   voiceless  40  30   30
#
#      [,1] [,2]
# low    -1   -1
# mid     1   -1
# high    0    2
#
# Coefficients:
#                    Estimate Std. Error t value Pr(>|t|)
# (Intercept)         17.0940     1.3038  13.111  < 2e-16 ***
# conditionvoiceless  52.5001     1.8354  28.604  < 2e-16 ***
# freq1               -1.5696     1.1082  -1.416  0.15827
# freq2               -2.1589     0.6696  -3.224  0.00148 **
#
# freq1 に対応する値 (mid - low)/2            : -1.5696
# freq2 に対応する値 (high - (low+mid)/2)/3   : -2.1589
#
# 【出力の読み方】
# ・1問目は数字が答えそのものである。treatment codingの切片22.867は
#   voicedの平均と一致し、sum codingの切片49.117は2群の平均の平均と一致する。
#   **切片は「予測変数がすべて0のときの予測値」であり、コーディングを
#   変えるとその「0」が指す場所が動く**——これが切片の意味が変わる理由である。
# ・クロス集計表で、どの頻度帯もvoicedとvoicelessが同数（40/40、30/30、30/30）に
#   なっている。**帯によって有声語の割合が違うと、頻度の効果と条件の効果が
#   混ざって分離できなくなる。** 係数を読む前に必ずこの表を見る。
# ・Helmertのコントラスト行列は列ごとに読む。第1列は low=-1, mid=+1, high=0
#   なのでhighを除外してmidとlowを比べる。第2列は low=-1, mid=-1, high=+2 で、
#   highを「lowとmidの平均」と比べる。**各列の合計が0**であること、
#   そして2つの列の積和も0になること（＝互いに直交）が要点である。
#   直交していると、係数どうしが影響し合わずに独立して解釈できる。
# ・係数の大きさは差そのものではなく、**割り算された値**である点に注意する。
#   最後の2行の手計算が、それを数値で証明している。
#     freq1 = (midの平均 − lowの平均) / 2         = -1.5696
#     freq2 = (highの平均 − (low+mid)の平均) / 3  = -2.1589
#   分母は「その比較に関わる水準の数」（1対1なら2、2対1なら3）である。
#   したがって**実際の差を報告したいときは係数を2倍・3倍する**必要がある。
#   mid−low = -3.14ms、high−(low,midの平均) = -6.48ms となる。
# ・p値を見ると、freq2は有意（0.00148）だがfreq1は有意でない（0.158）。
#   生成時にはmidで-6ms、highで-12msという効果を入れてあるので、
#   「lowとmidの差は検出できず、highの落ち込みだけ検出できた」ことになる。
#   帯あたりの語数が6〜8語しかなく、単語ごとのばらつき（第1章で入れたsd=3）が
#   帯の平均を数msずつ動かすためである。**入れた効果が必ず検出されるわけではない。**
# ・さらに重要な注意: このlm()は200行すべてを独立として扱っているが、
#   freqは**語の性質**なので、実質的な情報量は20語分しかない。
#   第3章で見た疑似反復と同じ構図である。単語ごとの効果を正しく検定するには、
#   第5章の (1 | word) を入れた混合効果モデルが必要になる。
